# Notebook 04 — Concolic Exploration

Runs Algorithm 1 (Concolic Perturbation Radius Estimation) on all three
trained networks and saves per-sample results for comparison.

This is the **core experimental notebook** — it produces the primary results
of the paper.

| Model | Norm | Samples | Est. Time |
|---|---|---|---|
| SmallMLP (MNIST) | L∞ | 100 | ~15 min |
| MediumMLP (MNIST) | L∞ | 100 | ~30 min |
| LargeMLP (CIFAR-10) | L∞ | 100 | ~45 min |

**What we measure per sample:**
- `ε_upper` — smallest adversarial perturbation found (constructive upper bound)
- `ε_lower` — activation-margin heuristic lower bound
- `gap` — ε_upper − ε_lower (bound tightness)
- `runtime` — wall-clock time per sample
- `n_iterations` — solver calls made

**Requires:** all three `.pt` model files from Notebook 01

**Outputs saved to:**
```
results/concolic_small_mnist.json
results/concolic_medium_mnist.json
results/concolic_large_cifar.json
results/concolic_summary.json
```

## 0 — Install dependencies

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'scipy', 'tqdm', 'numpy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)
import torch
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.12.0+cpu


## 1 — Imports & configuration

In [2]:
import sys, os, time
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.network_definitions import load_model
from utils.concolic_engine import ConcolicExplorer, run_concolic_batch
from utils.metrics import (
    compute_robustness_stats,
    get_correctly_classified_samples,
    save_results, print_table,
)

MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE    = 'cpu'
SEED      = 42
N_SAMPLES = 100

# ── concolic hyperparameters ──────────────────────────────────────────────────
# These match the paper's Algorithm 1 description
NORM       = 'linf'   # L∞ norm — matches FGSM/PGD for fair comparison
MAX_ITER   = 150      # N: solver calls per input
MAX_RADIUS = 1.0      # R: initial ε_upper

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Norm       : {NORM}')
print(f'Max iter   : {MAX_ITER}')
print(f'Max radius : {MAX_RADIUS}')
print(f'Samples    : {N_SAMPLES} per model')

Norm       : linf
Max iter   : 150
Max radius : 1.0
Samples    : 100 per model


## 2 — Load datasets

In [3]:
def dataset_to_numpy(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False,
                        num_workers=0)
    X, y = next(iter(loader))
    return X.view(X.size(0), -1).numpy(), y.numpy()

# MNIST
mnist_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_test = datasets.MNIST(DATA_DIR, train=False, download=True,
                             transform=mnist_tf)
X_mnist, y_mnist = dataset_to_numpy(mnist_test)
print(f'MNIST  : {X_mnist.shape}')

# CIFAR-10
cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
cifar_test = datasets.CIFAR10(DATA_DIR, train=False, download=True,
                               transform=cifar_tf)
X_cifar, y_cifar = dataset_to_numpy(cifar_test)
print(f'CIFAR-10: {X_cifar.shape}')

MNIST  : (10000, 784)
CIFAR-10: (10000, 3072)


## 3 — Helper: run and save

In [4]:
def run_and_save(model_name, model_path, X_data, y_data, save_path):
    """
    Full pipeline for one model:
      1. Load model
      2. Select N_SAMPLES correctly-classified inputs
      3. Run concolic exploration on each
      4. Compute aggregate stats
      5. Save to JSON
    """
    print(f'\n{"═"*60}')
    print(f'  {model_name}')
    print(f'{"═"*60}')

    # load
    model = load_model(str(model_path))
    print(f'  Architecture  : {model.hidden_dims}')
    print(f'  ReLU neurons  : {model.n_relu_neurons()}')

    # sample selection
    X_sel, y_sel = get_correctly_classified_samples(
        model, X_data, y_data, N_SAMPLES, seed=SEED
    )
    print(f'  Samples       : {len(X_sel)}')

    # run concolic exploration
    explorer = ConcolicExplorer(
        model,
        norm       = NORM,
        max_iter   = MAX_ITER,
        max_radius = MAX_RADIUS,
        verbose    = False,
    )

    per_sample = []
    bar = tqdm(zip(X_sel, y_sel), total=len(X_sel),
               desc=model_name, unit='sample')

    for x, label in bar:
        result = explorer.run(x, int(label))
        per_sample.append({
            'eps_upper'    : result.eps_upper,
            'eps_lower'    : result.eps_lower,
            'gap'          : result.eps_upper - result.eps_lower,
            'adv_found'    : result.adversarial_x is not None,
            'n_iterations' : result.n_iterations,
            'runtime_sec'  : result.runtime_sec,
        })
        bar.set_postfix(
            upper=f'{result.eps_upper:.3f}',
            lower=f'{result.eps_lower:.3f}',
            gap  =f'{result.eps_upper - result.eps_lower:.3f}',
            t    =f'{result.runtime_sec:.1f}s',
        )

    # aggregate
    uppers  = np.array([r['eps_upper']    for r in per_sample])
    lowers  = np.array([r['eps_lower']    for r in per_sample])
    gaps    = np.array([r['gap']          for r in per_sample])
    times   = np.array([r['runtime_sec']  for r in per_sample])
    iters   = np.array([r['n_iterations'] for r in per_sample])
    found   = np.array([r['adv_found']    for r in per_sample])

    stats = {
        'model_name'                : model_name,
        'n_relu_neurons'            : model.n_relu_neurons(),
        'n_samples'                 : len(per_sample),
        # upper bound
        'mean_eps_upper'            : float(np.mean(uppers)),
        'std_eps_upper'             : float(np.std(uppers)),
        'median_eps_upper'          : float(np.median(uppers)),
        # lower bound
        'mean_eps_lower'            : float(np.mean(lowers)),
        'std_eps_lower'             : float(np.std(lowers)),
        'median_eps_lower'          : float(np.median(lowers)),
        # gap
        'mean_gap'                  : float(np.mean(gaps)),
        'std_gap'                   : float(np.std(gaps)),
        # runtime
        'mean_runtime_sec'          : float(np.mean(times)),
        'total_runtime_sec'         : float(np.sum(times)),
        # iterations
        'mean_iterations'           : float(np.mean(iters)),
        # adversarial success
        'fraction_adversarial_found': float(np.mean(found)),
    }

    output = {'per_sample': per_sample, 'stats': stats}
    save_results(output, str(save_path))

    print(f'\n  ε_upper : {stats["mean_eps_upper"]:.4f} ± {stats["std_eps_upper"]:.4f}')
    print(f'  ε_lower : {stats["mean_eps_lower"]:.4f} ± {stats["std_eps_lower"]:.4f}')
    print(f'  gap     : {stats["mean_gap"]:.4f}')
    print(f'  adv%    : {stats["fraction_adversarial_found"]*100:.1f}%')
    print(f'  time/s  : {stats["mean_runtime_sec"]:.2f}s')
    print(f'  iters   : {stats["mean_iterations"]:.1f}')

    return output

print('Pipeline defined.')

Pipeline defined.


## 4 — Run on SmallMLP (MNIST)
**Expected time: ~15 min**

In [5]:
results_small = run_and_save(
    model_name = 'SmallMLP-MNIST',
    model_path = MODELS_DIR / 'small_mnist.pt',
    X_data     = X_mnist,
    y_data     = y_mnist,
    save_path  = RESULTS_DIR / 'concolic_small_mnist.json',
)


════════════════════════════════════════════════════════════
  SmallMLP-MNIST
════════════════════════════════════════════════════════════
  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}
  Architecture  : [64, 64]
  ReLU neurons  : 128
  Samples       : 100


SmallMLP-MNIST: 100%|███████████████| 100/100 [00:03<00:00, 25.56sample/s, gap=0.069, lower=0.184, t=0.0s, upper=0.253]


  Saved results → D:\concolic_exploration\results\concolic_small_mnist.json

  ε_upper : 0.1743 ± 0.0684
  ε_lower : 0.1395 ± 0.0604
  gap     : 0.0347
  adv%    : 100.0%
  time/s  : 0.04s
  iters   : 133.6


## 5 — Run on MediumMLP (MNIST)
**Expected time: ~30 min**

In [6]:
results_medium = run_and_save(
    model_name = 'MediumMLP-MNIST',
    model_path = MODELS_DIR / 'medium_mnist.pt',
    X_data     = X_mnist,
    y_data     = y_mnist,
    save_path  = RESULTS_DIR / 'concolic_medium_mnist.json',
)


════════════════════════════════════════════════════════════
  MediumMLP-MNIST
════════════════════════════════════════════════════════════
  Loaded ← D:\concolic_exploration\models\medium_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-256-256-256-10', 'best_test_acc': 0.9853, 'n_relu_neurons': 768, 'n_params': 335114}
  Architecture  : [256, 256, 256]
  ReLU neurons  : 768
  Samples       : 100


MediumMLP-MNIST: 100%|██████████████| 100/100 [00:05<00:00, 19.14sample/s, gap=0.000, lower=0.445, t=0.0s, upper=0.445]


  Saved results → D:\concolic_exploration\results\concolic_medium_mnist.json

  ε_upper : 0.2407 ± 0.1010
  ε_lower : 0.2291 ± 0.1021
  gap     : 0.0116
  adv%    : 100.0%
  time/s  : 0.05s
  iters   : 94.7


## 6 — Run on LargeMLP (CIFAR-10)
**Expected time: ~45 min**

In [7]:
results_large = run_and_save(
    model_name = 'LargeMLP-CIFAR10',
    model_path = MODELS_DIR / 'large_cifar.pt',
    X_data     = X_cifar,
    y_data     = y_cifar,
    save_path  = RESULTS_DIR / 'concolic_large_cifar.json',
)


════════════════════════════════════════════════════════════
  LargeMLP-CIFAR10
════════════════════════════════════════════════════════════
  Loaded ← D:\concolic_exploration\models\large_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'architecture': '3072-512-512-512-512-10', 'best_test_acc': 0.5593, 'n_relu_neurons': 2048, 'n_params': 2366474}
  Architecture  : [512, 512, 512, 512]
  ReLU neurons  : 2048
  Samples       : 100


LargeMLP-CIFAR10: 100%|█████████████| 100/100 [00:39<00:00,  2.55sample/s, gap=0.002, lower=0.010, t=0.3s, upper=0.012]

  Saved results → D:\concolic_exploration\results\concolic_large_cifar.json

  ε_upper : 0.0612 ± 0.0462
  ε_lower : 0.0471 ± 0.0370
  gap     : 0.0141
  adv%    : 100.0%
  time/s  : 0.39s
  iters   : 143.0


## 7 — Cross-model summary

In [8]:
summary = {
    'small_mnist'  : results_small['stats'],
    'medium_mnist' : results_medium['stats'],
    'large_cifar'  : results_large['stats'],
    'config': {
        'norm'       : NORM,
        'max_iter'   : MAX_ITER,
        'max_radius' : MAX_RADIUS,
        'n_samples'  : N_SAMPLES,
        'seed'       : SEED,
    }
}
save_results(summary, str(RESULTS_DIR / 'concolic_summary.json'))

# ── print table ───────────────────────────────────────────────────────────────
print()
print('═'*75)
print('  CONCOLIC EXPLORATION SUMMARY')
print('═'*75)
print(f'  {"Model":<22} {"ReLU n":>6} {"ε_upper":>10} {"ε_lower":>10} '
      f'{"gap":>8} {"time/s":>8}')
print('─'*75)
for name, s in [('small_mnist',  results_small['stats']),
                 ('medium_mnist', results_medium['stats']),
                 ('large_cifar',  results_large['stats'])]:
    print(f"  {name:<22} {s['n_relu_neurons']:>6} "
          f"{s['mean_eps_upper']:>10.4f} "
          f"{s['mean_eps_lower']:>10.4f} "
          f"{s['mean_gap']:>8.4f} "
          f"{s['mean_runtime_sec']:>8.2f}")
print('═'*75)
print()
print('  Key: ε_upper = constructive upper bound (adversarial example found)')
print('       ε_lower = activation-margin heuristic lower bound')
print('       gap     = uncertainty interval [ε_lower, ε_upper]')
print()
print('  Next step → run 05_analysis_plots.ipynb')

  Saved results → D:\concolic_exploration\results\concolic_summary.json

═══════════════════════════════════════════════════════════════════════════
  CONCOLIC EXPLORATION SUMMARY
═══════════════════════════════════════════════════════════════════════════
  Model                  ReLU n    ε_upper    ε_lower      gap   time/s
───────────────────────────────────────────────────────────────────────────
  small_mnist               128     0.1743     0.1395   0.0347     0.04
  medium_mnist              768     0.2407     0.2291   0.0116     0.05
  large_cifar              2048     0.0612     0.0471   0.0141     0.39
═══════════════════════════════════════════════════════════════════════════

  Key: ε_upper = constructive upper bound (adversarial example found)
       ε_lower = activation-margin heuristic lower bound
       gap     = uncertainty interval [ε_lower, ε_upper]

  Next step → run 05_analysis_plots.ipynb


## 8 — Heuristic sensitivity analysis

Test how the neuron selection heuristic affects bound quality.
We compare `nearest-boundary-first` (default) against `random` selection
on the SmallMLP with fewer iterations.

In [9]:
print('Running heuristic sensitivity analysis on SmallMLP...')
print('(50 samples, 50 iterations each — ~5 min)\n')

from utils.concolic_engine import ActivationRegion, BoundaryOptimiser, ConcolicResult
import random as rnd

class RandomConcolicExplorer(ConcolicExplorer):
    """Variant that selects neurons randomly instead of nearest-boundary-first.
    Uses v2 API: optimiser.solve(neuron, x, eps_upper, model, original_c)
    """
    def run(self, x, true_label):
        import time
        t_start = time.time()
        c = self._predict(x)
        if c != true_label:
            return ConcolicResult(0.0, 0.0, None, 0, time.time()-t_start)

        region  = ActivationRegion(self.model, x)
        neurons = list(enumerate(region.neurons))
        rnd.shuffle(neurons)   # random order — key difference from NBF

        # gradient lower bound (same as main explorer)
        eps_lower = region.gradient_lower_bound(self.model, x, c, self.norm)
        eps_upper = self.max_radius
        best_adv  = None
        log       = []

        for iteration in range(self.max_iter):
            neuron_idx, neuron = neurons[iteration % len(neurons)]
            # v2 API: pass model and original_c, not region.neurons and neuron_idx
            delta = self.optimiser.solve(
                neuron, x, eps_upper, self.model, c
            )
            if delta is not None:
                norm_delta = self._norm(delta)
                c_prime    = self._predict(x + delta)
                if c_prime != c and norm_delta < eps_upper:
                    eps_upper = norm_delta
                    best_adv  = (x + delta).copy()
            if eps_upper - eps_lower < 1e-5:
                break
            log.append({})

        if best_adv is not None:
            eps_lower = max(eps_lower, eps_upper * 0.5)
        eps_lower = min(eps_lower, eps_upper)

        return ConcolicResult(
            eps_upper, eps_lower, best_adv,
            len(log), time.time()-t_start
        )


small_model = load_model(str(MODELS_DIR / 'small_mnist.pt'))
X_sens, y_sens = get_correctly_classified_samples(
    small_model, X_mnist, y_mnist, 50, seed=SEED
)

# nearest-boundary-first
nbf_results = run_concolic_batch(
    small_model, X_sens, y_sens,
    norm='linf', max_iter=50, max_radius=1.0, verbose=False
)
nbf_uppers = np.array([r.eps_upper for r in nbf_results])

# random selection
rnd.seed(SEED)
rand_explorer = RandomConcolicExplorer(
    small_model, norm='linf', max_iter=50, max_radius=1.0
)
rand_results = [rand_explorer.run(x, int(l)) for x, l in
                tqdm(zip(X_sens, y_sens), total=50, desc='Random heuristic')]
rand_uppers  = np.array([r.eps_upper for r in rand_results])

sensitivity = {
    'nearest_boundary_first': {
        'mean_eps_upper': float(np.mean(nbf_uppers)),
        'std_eps_upper' : float(np.std(nbf_uppers)),
    },
    'random_selection': {
        'mean_eps_upper': float(np.mean(rand_uppers)),
        'std_eps_upper' : float(np.std(rand_uppers)),
    },
}
save_results(sensitivity, str(RESULTS_DIR / 'heuristic_sensitivity.json'))

print(f'\n  Nearest-boundary-first : ε_upper = {np.mean(nbf_uppers):.4f} ± {np.std(nbf_uppers):.4f}')
print(f'  Random selection       : ε_upper = {np.mean(rand_uppers):.4f} ± {np.std(rand_uppers):.4f}')
improvement = (np.mean(rand_uppers) - np.mean(nbf_uppers)) / np.mean(rand_uppers) * 100
print(f'  Improvement from heuristic: {improvement:.1f}% tighter upper bound')

Running heuristic sensitivity analysis on SmallMLP...
(50 samples, 50 iterations each — ~5 min)

  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}
  [10/50] ε_upper=0.2823  ε_lower=0.1411  gap=0.1411  iters=50  time=0.04s
  [20/50] ε_upper=0.1266  ε_lower=0.0845  gap=0.0421  iters=50  time=0.05s
  [30/50] ε_upper=0.1568  ε_lower=0.1366  gap=0.0203  iters=50  time=0.04s
  [40/50] ε_upper=0.2172  ε_lower=0.1749  gap=0.0423  iters=50  time=0.02s
  [50/50] ε_upper=0.2779  ε_lower=0.1390  gap=0.1390  iters=50  time=0.02s


Random heuristic: 100%|████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 36.61it/s]

  Saved results → D:\concolic_exploration\results\heuristic_sensitivity.json

  Nearest-boundary-first : ε_upper = 0.1794 ± 0.0795
  Random selection       : ε_upper = 0.1803 ± 0.0682
  Improvement from heuristic: 0.5% tighter upper bound
